In [6]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split

from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor

from sklearn.metrics import (
    mean_squared_error,
    mean_absolute_error,
    r2_score
)

import mlflow
import mlflow.sklearn
import mlflow.xgboost
import dagshub

In [7]:
dagshub.init(
    repo_owner="radhakrishn-an",
    repo_name="Boston-MLflow",
    mlflow=True
)

Initialized MLflow to track repo "radhakrishn-an/Boston-MLflow"

Repository radhakrishn-an/Boston-MLflow initialized!

In [8]:
mlflow.set_experiment(
    "Boston Housing Regression"
)

2026/08/06 12:53:53 INFO mlflow.tracking.fluent: Experiment with name 'Boston Housing Regression' does not exist. Creating a new experiment.


<Experiment: artifact_location='mlflow-artifacts:/77480c2f887a48a5b2f7b7b5525afce5', creation_time=1786001033864, effective_trace_archival_retention=None, experiment_id='0', last_update_time=1786001033864, lifecycle_stage='active', name='Boston Housing Regression', tags={}, trace_location=None, workspace='default'>

In [14]:
boston = fetch_openml(
    name="boston",
    version=1,
    as_frame=True
)

X = boston.data.copy()
y = boston.target.astype(float)

# Convert category columns to integer codes
for col in X.select_dtypes(include=["category"]).columns:
    X[col] = X[col].cat.codes

print(X.dtypes)

CRIM       float64
ZN         float64
INDUS      float64
CHAS          int8
NOX        float64
RM         float64
AGE        float64
DIS        float64
RAD           int8
TAX        float64
PTRATIO    float64
B          float64
LSTAT      float64
dtype: object


In [15]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.30,
    random_state=42
)

In [16]:
models = [

    (
        "Linear Regression",
        LinearRegression()
    ),

    (
        "Random Forest",
        RandomForestRegressor(
            n_estimators=100,
            random_state=42
        )
    ),

    (
        "XGBoost",
        XGBRegressor(
            n_estimators=100,
            learning_rate=0.1,
            max_depth=4,
            random_state=42
        )
    )
]

In [17]:
reports = []
trained_models = []

for model_name, model in models:

    model.fit(X_train, y_train)

    predictions = model.predict(X_test)

    rmse = np.sqrt(
        mean_squared_error(
            y_test,
            predictions
        )
    )

    mae = mean_absolute_error(
        y_test,
        predictions
    )

    r2 = r2_score(
        y_test,
        predictions
    )

    reports.append(
        {
            "RMSE": rmse,
            "MAE": mae,
            "R2": r2
        }
    )

    trained_models.append(model)

    print("="*50)
    print(model_name)
    print("RMSE :", rmse)
    print("MAE  :", mae)
    print("R2   :", r2)

Linear Regression
RMSE : 4.8179735394911205
MAE  : 3.288398477695366
R2   : 0.6884726255119256
Random Forest
RMSE : 3.1179096732046547
MAE  : 2.0946907894736837
R2   : 0.869534869593374
XGBoost
RMSE : 3.2195166230594285
MAE  : 2.0973312409300555
R2   : 0.8608930791673363


In [18]:
for i, (model_name, model) in enumerate(models):

    report = reports[i]

    with mlflow.start_run(
        run_name=model_name
    ):

        mlflow.log_param(
            "Model",
            model_name
        )

        mlflow.log_params(
            model.get_params()
        )

        mlflow.log_metric(
            "RMSE",
            report["RMSE"]
        )

        mlflow.log_metric(
            "MAE",
            report["MAE"]
        )

        mlflow.log_metric(
            "R2",
            report["R2"]
        )

        if "XGBoost" in model_name:

            mlflow.xgboost.log_model(
                model,
                "model"
            )

        else:

            mlflow.sklearn.log_model(
                model,
                "model"
            )

        print(model_name, "Logged Successfully")

2026/08/06 13:01:41 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


Linear Regression Logged Successfully
🏃 View run Linear Regression at: https://dagshub.com/radhakrishn-an/Boston-MLflow.mlflow/#/experiments/0/runs/42290aabf7fa47b3937cd3250d51a2b8
🧪 View experiment at: https://dagshub.com/radhakrishn-an/Boston-MLflow.mlflow/#/experiments/0


2026/08/06 13:02:06 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


Random Forest Logged Successfully
🏃 View run Random Forest at: https://dagshub.com/radhakrishn-an/Boston-MLflow.mlflow/#/experiments/0/runs/f41d725d00084f2d9201c7186853302b
🧪 View experiment at: https://dagshub.com/radhakrishn-an/Boston-MLflow.mlflow/#/experiments/0


2026/08/06 13:02:40 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


XGBoost Logged Successfully
🏃 View run XGBoost at: https://dagshub.com/radhakrishn-an/Boston-MLflow.mlflow/#/experiments/0/runs/82fa2125741248aa8ec58ad30949fd50
🧪 View experiment at: https://dagshub.com/radhakrishn-an/Boston-MLflow.mlflow/#/experiments/0


In [19]:
best_index = np.argmax(
    [
        r["R2"]
        for r in reports
    ]
)

best_model_name = models[best_index][0]

best_model = trained_models[best_index]

best_report = reports[best_index]

print("Best Model :", best_model_name)

Best Model : Random Forest


In [20]:
with mlflow.start_run(
    run_name=f"Champion_{best_model_name}"
) as run:

    mlflow.log_param(
        "Model",
        best_model_name
    )

    mlflow.log_metric(
        "RMSE",
        best_report["RMSE"]
    )

    mlflow.log_metric(
        "MAE",
        best_report["MAE"]
    )

    mlflow.log_metric(
        "R2",
        best_report["R2"]
    )

    if "XGBoost" in best_model_name:

        mlflow.xgboost.log_model(
            best_model,
            name="model",
            registered_model_name="Boston_Housing_Best_Model"
        )

    else:

        mlflow.sklearn.log_model(
            best_model,
            name="model",
            registered_model_name="Boston_Housing_Best_Model"
        )

    run_id = run.info.run_id

print(run_id)

Successfully registered model 'Boston_Housing_Best_Model'.
2026/08/06 13:06:22 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: Boston_Housing_Best_Model, version 1
Created version '1' of model 'Boston_Housing_Best_Model'.


🏃 View run Champion_Random Forest at: https://dagshub.com/radhakrishn-an/Boston-MLflow.mlflow/#/experiments/0/runs/40cdf72976244745adf9ee4f1c30d3c2
🧪 View experiment at: https://dagshub.com/radhakrishn-an/Boston-MLflow.mlflow/#/experiments/0
40cdf72976244745adf9ee4f1c30d3c2
